In [3]:
from fetch_data import load_all
data = load_all()

income_records = data["hh_income_state"]
poverty_records = data["hh_poverty_state"]

In [4]:
## Assign group member focus state
from solution import get_member_state, STATE
matrix_num = [309466, 310176, 310283, 310415, 310567]
GROUP_STATE = get_member_state(matrix_num, STATE)
print("Group member focus state: ", GROUP_STATE)

Group member focus state:  ['Sarawak', 'Johor', 'Selangor', 'W.P. Putrajaya', 'Perlis']


In [13]:
import numpy as np
from solution import validate_records, SURVEY_YEARS

def analyze_dataset_records(records: list[dict], dataset_name: str, numeric_fields: list[str]):
    print(f"=== ANALYSIS REPORT: {dataset_name} ===")

    num_records = len(records)
    year = [int(str(rec["date"])[:4]) for rec in records]
    sorted_year = sorted(year)
    first_year = sorted_year[0]
    last_year = sorted_year[-1]
    distinct_year = len(set(sorted_year))
    print(f"1. Total record: {num_records}")
    print(f"   First survey year: {first_year}, Last survey year: {last_year}")
    print(f"   Distinct survey year: {distinct_year} -> {set(sorted_year)}")
    print()

    all_fields = list(records[0].keys())
    none_count = {field: sum(1 for rec in records if rec.get(field) is None ) for field in all_fields}
    print("2. None Values Count per Field:")
    for field, count in none_count.items():
        print(f"     - {field}: {count} None Value")
    print()

    print("3. Python Types in Numeric Fields:")
    for field in numeric_fields:
        found_type = set(type(rec.get(field)).__name__ for rec in records)
        print(f"   - {field}: {found_type}")
        
        non_num_count = sum(1 for rec in records if not isinstance(rec.get(field), (int, float)) and rec.get(field) is not None)
        if non_num_count > 0:
            print(f"      Inconsistency warning: Found {non_num_count} non-numeric/string values!")
            
    print("\n" + "="*50 + "\n")

analyze_dataset_records(income_records, "Income Dataset (hh_income_state)", ["income_mean", "income_median"])
analyze_dataset_records(poverty_records, "Poverty Dataset (hh_poverty_state)", ["poverty_absolute", "poverty_hardcore"])

filtered_income = [rec for rec in income_records if int(str(rec["date"])[:4]) in SURVEY_YEARS]
all_states = sorted(list(set(rec["state"] for rec in income_records)))

for yr in SURVEY_YEARS:
    states_in_year = set(rec["state"] for rec in filtered_income if int(str(rec["date"])[:4]) == yr)
    missing = set(all_states) - states_in_year
    if missing:
        raise ValueError(f"Income check failed! Year {yr} is missing states: {missing}")
print(f" Completeness Check Passed: All 16 states are present across all SURVEY_YEARS {SURVEY_YEARS}.")

=== ANALYSIS REPORT: Income Dataset (hh_income_state) ===
1. Total record: 319
   First survey year: 1970, Last survey year: 2024
   Distinct survey year: 22 -> {1970, 1974, 1976, 1979, 1984, 1987, 1989, 1992, 1995, 1997, 1999, 2002, 2004, 2007, 2009, 2012, 2014, 2016, 2019, 2020, 2022, 2024}

2. None Values Count per Field:
     - date: 0 None Value
     - state: 0 None Value
     - income_mean: 0 None Value
     - income_median: 11 None Value

3. Python Types in Numeric Fields:
   - income_mean: {'int'}
   - income_median: {'float', 'NoneType'}


=== ANALYSIS REPORT: Poverty Dataset (hh_poverty_state) ===
1. Total record: 310
   First survey year: 1970, Last survey year: 2024
   Distinct survey year: 21 -> {1970, 1976, 1979, 1984, 1987, 1989, 1992, 1995, 1997, 1999, 2002, 2004, 2007, 2009, 2012, 2014, 2016, 2019, 2020, 2022, 2024}

2. None Values Count per Field:
     - date: 0 None Value
     - state: 0 None Value
     - poverty_absolute: 2 None Value
     - poverty_hardcore: 39 Non